# DustyLM Quick Start: Chat with Dusty

Run the first two cells and start chatting with Dusty, a tiny SFT assistant model that talks like a small vacuum robot.

The notebook is split in two:

- **Quick Start:** the shortest path to a working chat loop.
- **Advanced: Under the Hood:** API shape, context management, and profile detection.


### 1. Download

Download the Dusty checkpoint and tokenizer from Hugging Face:


In [ ]:
# Install the required Hugging Face hub library if missing
!pip install -q uv huggingface_hub
!uv pip install --system -e .

from huggingface_hub import hf_hub_download
import sys
from pathlib import Path

# Add project root to path
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "dustylm").exists():
    sys.path.insert(0, str(REPO_ROOT))

print("Downloading Dusty checkpoint and tokenizer...")

# Automatically download the artifacts
checkpoint_path = hf_hub_download(
    repo_id="khordoo/dusty-8m-sft", filename="dusty8m_sft.pt"
)
tokenizer_path = hf_hub_download(
    repo_id="khordoo/dusty-8m-sft", filename="tokenizer.json"
)

print(f"Checkpoint ready: {checkpoint_path}")
print(f"Tokenizer ready: {tokenizer_path}")

### 2. Chat

This cell loads the model once, then immediately sends a few OpenAI-compatible chat completions. 


In [ ]:
import torch
from dustylm.inference import Inference

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

engine = Inference(
    checkpoint_path=checkpoint_path,
    tokenizer_path=tokenizer_path,
    device=device,
)


# Create a simple wrapper so we can loop through a few questions.
def chat(prompt):
    # We use OpenAI-compatible completion here.
    response = engine.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=64,
        temperature=0.6,
        top_p=0.8,
    )
    return response["choices"][0]["message"]["content"].strip()


prompts = [
    "who are you?",
    "what is under the sofa?",
    "are you afraid of the cat?",
]

for prompt in prompts:
    print(f"You> {prompt}")
    print(f"Dusty> {chat(prompt)}\n")


## Advanced: Under the Hood

The framework is doing more than printing text. This section shows how the inference engine handles API formatting, context limits, and profile detection while keeping the quick-start path short.


### OpenAI-Style Chat Completion

The public API accepts OpenAI-style `messages` and returns an OpenAI-like dictionary. This makes it easy to swap between local DustyLM inference and common chat-completion calling patterns.

In [ ]:
messages = [{"role": "user", "content": "where are you?"}]

response = engine.chat_completion(
    messages,
    max_tokens=64,
    temperature=0.8,
    top_p=0.8,
)

response


In [ ]:
assistant_text = response["choices"][0]["message"]["content"]
print("Dusty>", assistant_text)


### Tiny Models Need Short Context

Dusty is only about 8M parameters, so the default chat history window is intentionally tiny: `max_chat_turns=1`. That keeps the current user request from being diluted by old conversation tokens.

Larger SFT profiles, such as SmolLM2-based models, can use a larger window.

In [ ]:
print("Profile:", engine.profile_name)
print("Default max_chat_turns:", engine.spec.max_chat_turns)

history = [
    {"role": "system", "content": "Answer as Dusty, a tiny vacuum robot."},
    {"role": "user", "content": "what is under the couch?"},
    {"role": "assistant", "content": "crumbs. many crumbs."},
    {"role": "user", "content": "should you go there?"},
]

response = engine.chat_completion(history, max_tokens=64)
print("Default window:", response["choices"][0]["message"]["content"])

response = engine.chat_completion(history, max_tokens=64, max_chat_turns=2)
print("Two-turn window:", response["choices"][0]["message"]["content"])


### Profile Detection by Sidecar Config or Weight Sniffing

DustyLM keeps `.pt` checkpoints as plain PyTorch weights. Runtime profile detection happens outside the checkpoint:

1. Look for `<checkpoint_stem>.json` or `config.json` with `profile_name`.
2. If no config exists, inspect CPU-loaded tensor keys.
3. Fall back to `sft_dusty8m`.

Dusty-style checkpoints contain keys such as `embed.weight` and `attention.qkv_proj`. SmolLM2-style checkpoints contain keys such as `embed_tokens.weight`, `gate_proj`, and `up_proj`.

In [ ]:
from dustylm.checkpoint import resolve_profile_name_for_checkpoint, load_state_dict

profile_name = resolve_profile_name_for_checkpoint(
    checkpoint_path,
    default_profile="sft_dusty8m",
    mode="chat",
)
print("Resolved profile:", profile_name)

state_dict = load_state_dict(checkpoint_path, map_location="cpu")
keys = list(state_dict.keys())[:12]
print("First checkpoint keys:")
for key in keys:
    print(" -", key)


### A Small Convenience Wrapper

For scripts or demos, you can hide the response dictionary and return just the generated assistant text.

In [ ]:
def chat(prompt, *, max_tokens=64, temperature=0.8, top_p=0.8):
    response = engine.chat_completion(
        [{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    return response["choices"][0]["message"]["content"].strip()


for prompt in ["hi dusty", "are you scared?", "what do you dream about?", "go charge"]:
    print(f"You> {prompt}")
    print(f"Dusty> {chat(prompt)}\n")


### Want an Interactive Chat Loop?

The notebook avoids a `while True` input loop so it stays easy to run top-to-bottom. For interactive chat, use the CLI instead:

```bash
make chat
```

That target runs:

```bash
uv run python -m dustylm.inference
```
